# US Superstore - Business Intelligence Report

This notebook delivers a full diagnostic BI analysis for a national retailer using the US Superstore dataset.

## Objectives
- Explore business performance across time, geography, product, and discount strategy
- Build interactive and explanatory visualizations
- Produce strategic recommendations for decision-makers

## 1) Data Scoping and Preparation

Before running this notebook, place the Kaggle file in this folder and rename it to `superstore_dataset.csv`.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

In [ ]:
# Load dataset
data_path = 'superstore_dataset.csv'
if not os.path.exists(data_path):
    raise FileNotFoundError(
        "Dataset not found. Add 'superstore_dataset.csv' to this folder: Week5/Day4/Daily_challenge"
    )

df = pd.read_csv(data_path)

print('Dataset Shape:', df.shape)
print('\nColumn Names:')
print(df.columns.tolist())
display(df.head())
display(df.describe(include='all').T.head(15))
print('\nMissing values per column:')
print(df.isnull().sum())

### Cleaning Decisions (Justification)
- **Duplicates:** dropped to avoid counting transactions more than once.
- **Postal Code:** if missing, set to `0` because it is an identifier (not a continuous metric used in modeling here).
- **Other missing values:** small number of rows are removed with `dropna()` to keep analysis robust and transparent.
- **Dates:** converted to `datetime` to enable monthly/yearly trend diagnostics.

In [ ]:
# Handle duplicates
duplicate_count = df.duplicated().sum()
print('Duplicate rows:', duplicate_count)
if duplicate_count > 0:
    df = df.drop_duplicates()

# Handle missing values
print('\nMissing values before cleaning:')
print(df.isnull().sum())

if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

# Drop remaining null rows (if any) after handling key identifier
df = df.dropna()

# Fix date types
date_columns = ['Order Date', 'Ship Date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Drop rows where date parsing failed
df = df.dropna(subset=[col for col in date_columns if col in df.columns])

print('\nData types after conversion:')
print(df[date_columns].dtypes)
print('\nMissing values after cleaning:')
print(df.isnull().sum())
print('\nFinal dataset shape:', df.shape)

In [ ]:
# Feature engineering
df['Profit Margin'] = np.where(df['Sales'] != 0, (df['Profit'] / df['Sales']) * 100, np.nan)
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print('New features created:')
display(df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head())

## 2) Deep-Dive Exploratory Analysis (Matplotlib)

In [ ]:
# Time-series preparation
monthly_sales = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    plt.figure(figsize=(12, 6))

    if category == 'All':
        total_monthly = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(total_monthly.index.to_timestamp(), total_monthly.values,
                 marker='o', linewidth=2, markersize=4)
        plt.title('Monthly Sales Trend - All Categories', fontsize=16, fontweight='bold')
    else:
        category_data = monthly_sales[monthly_sales['Category'] == category]
        plt.plot(category_data['Date'], category_data['Sales'],
                 marker='o', linewidth=2, markersize=4)
        plt.title(f'Monthly Sales Trend - {category}', fontsize=16, fontweight='bold')

    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Sales ($)', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

categories = ['All'] + sorted(df['Category'].dropna().unique().tolist())
category_dropdown = Dropdown(options=categories, value='All', description='Category:')
interact(plot_monthly_sales, category=category_dropdown)

In [ ]:
# Geographic sales performance
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10):
    plt.figure(figsize=(12, max(6, top_n * 0.4)))
    top_states = state_sales.tail(top_n)

    bars = plt.barh(range(len(top_states)), top_states.values, color='steelblue')
    plt.yticks(range(len(top_states)), top_states.index)
    plt.xlabel('Total Sales ($)', fontsize=12)
    plt.ylabel('State', fontsize=12)
    plt.title(f'Top {top_n} States by Sales Performance', fontsize=16, fontweight='bold')

    for i, (state, value) in enumerate(top_states.items()):
        plt.text(value + top_states.max() * 0.01, i, f'${value:,.0f}', va='center', fontsize=10)

    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'Total states analyzed: {len(state_sales)}')
    print(f'Top {top_n} states represent: ${top_states.sum():,.0f} in sales')

top_n_slider = IntSlider(min=5, max=min(25, len(state_sales)), value=min(10, len(state_sales)), description='Top N States:')
interact(plot_top_states, top_n=top_n_slider)

## 3) Communicating Insights (Seaborn)

In [ ]:
# Top 10 most profitable products
product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 8))
ax = sns.barplot(x=product_profit.values, y=product_profit.index, palette='viridis', orient='h')

plt.title('Top 10 Most Profitable Products\nExecutive Summary - Product Performance Analysis',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Total Profit ($)', fontsize=12, fontweight='bold')
plt.ylabel('Product Name', fontsize=12, fontweight='bold')

for i, (product, profit) in enumerate(product_profit.items()):
    ax.text(profit + product_profit.max() * 0.01, i, f'${profit:,.0f}',
            va='center', fontweight='bold', fontsize=10)

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('Key Insights:')
print(f'• Most profitable product generates: ${product_profit.iloc[0]:,.0f}')
print(f'• Top 10 products contribute: ${product_profit.sum():,.0f} total profit')
print(f'• Average profit per top product: ${product_profit.mean():,.0f}')

In [ ]:
# Discount vs Profit scatter + trend
plt.figure(figsize=(14, 8))

sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6, s=50)
sns.regplot(data=df, x='Discount', y='Profit', scatter=False, color='red',
            line_kws={'linewidth': 2, 'linestyle': '--'})

plt.title('Discount Strategy Analysis: Impact on Profitability by Category',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Discount Rate', fontsize=12, fontweight='bold')
plt.ylabel('Profit ($)', fontsize=12, fontweight='bold')
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
plt.text(df['Discount'].max() * 0.55, 50, 'Break-even line', fontsize=10, alpha=0.7)

plt.grid(True, alpha=0.3)
plt.legend(title='Product Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print('Discount Analysis Insights:')
high_discount = df[df['Discount'] > 0.2]
print(f'• Transactions with >20% discount: {len(high_discount):,}')
print(f'• Average profit for high discounts: ${high_discount[
].mean():.2f}')
loss_rate = (high_discount['Profit'] < 0).mean() * 100 if len(high_discount) > 0 else 0
print(f'• Percentage of high-discount sales with losses: {loss_rate:.1f}%')

print('\nCategory-specific high-discount profitability:')
for category in sorted(df['Category'].unique()):
    high_disc_cat = df[(df['Category'] == category) & (df['Discount'] > 0.2)]
    if len(high_disc_cat) > 0:
        avg_profit = high_disc_cat['Profit'].mean()
        print(f'• {category}: Average profit at >20% discount = ${avg_profit:.2f}')

## 4) Methodology and Tooling Review

### Matplotlib vs Seaborn (Comparative Evaluation)
- **Matplotlib** excels when precise control is required (custom annotations, widget-driven interaction, layout tuning).
- **Seaborn** excels for rapid explanatory visuals with cleaner defaults and built-in statistical layers (like regression).
- In practice, combining both gives the best analyst workflow: Matplotlib for interactive diagnostics, Seaborn for communication-ready charts.

**Recommendation:**
For rapid exploration, I use **Matplotlib** for control and interactive widget integration. For stakeholder-facing reporting, I use **Seaborn** for clearer aesthetics and built-in statistical communication.

In [ ]:
# Lightweight speed comparison
import time

print('=== LIBRARY COMPARISON ANALYSIS ===')

start = time.time()
plt.figure(figsize=(8, 6))
plt.plot(df.groupby('Order Year')['Sales'].sum())
plt.close()
matplotlib_time = time.time() - start

start = time.time()
plt.figure(figsize=(8, 6))
sns.lineplot(data=df.groupby('Order Year')['Sales'].sum().reset_index(), x='Order Year', y='Sales')
plt.close()
seaborn_time = time.time() - start

print(f'Matplotlib basic plot: {matplotlib_time:.4f} seconds')
print(f'Seaborn equivalent: {seaborn_time:.4f} seconds')

## 5) Executive Summary and Strategic Recommendations

In [ ]:
# Automated executive summary metrics
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
overall_profit_margin = (total_profit / total_sales) * 100 if total_sales != 0 else np.nan

top_state = state_sales.index[-1]
top_state_sales = state_sales.iloc[-1]
geo_concentration = (state_sales.tail(5).sum() / total_sales) * 100 if total_sales != 0 else np.nan

top_category = df.groupby('Category')['Sales'].sum().sort_values(ascending=False).index[0]
high_discount = df[df['Discount'] > 0.2]
high_discount_loss_rate = (high_discount['Profit'] < 0).mean() * 100 if len(high_discount) > 0 else 0

print('=== EXECUTIVE SUMMARY - KEY FINDINGS ===')
print()
print('BUSINESS PERFORMANCE:')
print(f'• Total Revenue: ${total_sales:,.0f}')
print(f'• Total Profit: ${total_profit:,.0f}')
print(f'• Overall Profit Margin: {overall_profit_margin:.1f}%')
print()
print('GEOGRAPHIC PERFORMANCE:')
print(f'• Top performing state: {top_state} (${top_state_sales:,.0f})')
print(f'• Geographic concentration: Top 5 states = {geo_concentration:.1f}% of sales')
print()
print('PRODUCT PERFORMANCE:')
print(f'• Leading category by sales: {top_category}')
print(f'• Most profitable product: {product_profit.index[0]}')
print()
print('DISCOUNT STRATEGY:')
print(f'• {high_discount_loss_rate:.1f}% of >20% discounts result in losses')
print('• Recommended max discount threshold: 20% (with manager approval for exceptions)')

### Final Strategic Recommendations
- Limit standard discounts to **20% maximum**, especially for low-margin product lines.
- Prioritize inventory and campaign investment in top-performing states while developing recovery plans for weak regions.
- Protect and promote top profitable products with premium placement and bundle strategies.
- Track category-specific discount elasticity monthly to balance growth and margin.

## Optional Advanced Challenge
Run the cell below for a compact 2x2 dashboard combining key visuals.

In [ ]:
def create_dashboard():
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    # 1) Monthly sales trend
    monthly_total = df.groupby('Order Month-Year')['Sales'].sum()
    ax1.plot(monthly_total.index.to_timestamp(), monthly_total.values, marker='o')
    ax1.set_title('Monthly Sales Trend')
    ax1.tick_params(axis='x', rotation=45)

    # 2) Category performance
    category_sales = df.groupby('Category')['Sales'].sum()
    ax2.bar(category_sales.index, category_sales.values, color='teal')
    ax2.set_title('Sales by Category')

    # 3) State performance (top 10)
    top_10_states = state_sales.tail(10)
    ax3.barh(range(len(top_10_states)), top_10_states.values, color='slateblue')
    ax3.set_yticks(range(len(top_10_states)))
    ax3.set_yticklabels(top_10_states.index)
    ax3.set_title('Top 10 States by Sales')

    # 4) Discount vs Profit
    for category in sorted(df['Category'].unique()):
        cat_data = df[df['Category'] == category]
        ax4.scatter(cat_data['Discount'], cat_data['Profit'], label=category, alpha=0.6)
    ax4.set_xlabel('Discount')
    ax4.set_ylabel('Profit')
    ax4.set_title('Discount vs Profit by Category')
    ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax4.legend()

    plt.tight_layout()
    plt.show()

create_dashboard()